# 06 - Custom Modelfile Workflow

This notebook covers GGUF + Modelfile customization for your stack and validates model behavior.

It focuses on your chart pattern where `__GGUF_PATH__` is replaced at render time and loaded into Ollama.


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
from pathlib import Path

import requests
import yaml

PROJECT_ROOT = Path("/media/waqasm86/External1/Project-Nvidia-Office/Project-Llamatelemetry/langchain-kubernetes-jupyterlab/llm-observability-stack")
MODEFILE_PATH = PROJECT_ROOT / "Modelfile.gemma-3-1b-it-gguf"
VALUES_PATH = PROJECT_ROOT / "values.local-k3s.yaml"
OLLAMA_URL = "http://localhost:11434"
OLLAMA_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/ollama 11434:11434"
BASE_MODEL = "gemma3-1b-it-gguf-local"
CUSTOM_MODEL = "gemma3-1b-it-gguf-local-tuned"


def ollama_available(url: str = OLLAMA_URL) -> bool:
    try:
        r = requests.get(f"{url}/api/tags", timeout=3)
        return r.status_code < 500
    except Exception:
        return False


def fetch_model_names(url: str = OLLAMA_URL) -> list[str]:
    try:
        r = requests.get(f"{url}/api/tags", timeout=10)
        r.raise_for_status()
        payload = r.json() or {}
    except Exception:
        return []
    return sorted({model.get("name") for model in (payload.get("models") or []) if model.get("name")})


OLLAMA_AVAILABLE = ollama_available()
AVAILABLE_MODELS = fetch_model_names() if OLLAMA_AVAILABLE else []
CUSTOM_MODEL_AVAILABLE = CUSTOM_MODEL in AVAILABLE_MODELS
print("Modelfile path:", MODEFILE_PATH)
print("Values path   :", VALUES_PATH)
print("OLLAMA_AVAILABLE:", OLLAMA_AVAILABLE)
print("CUSTOM_MODEL_AVAILABLE:", CUSTOM_MODEL_AVAILABLE)
if AVAILABLE_MODELS:
    print("Discovered models:", ", ".join(AVAILABLE_MODELS))
if not OLLAMA_AVAILABLE:
    print("Local Ollama endpoint is not reachable on localhost:11434.")
    print("In another terminal run:")
    print(f"  {OLLAMA_PORT_FORWARD_CMD}")


## Inspect Existing Modelfile


In [ ]:
modelfile_text = MODEFILE_PATH.read_text()
print(modelfile_text)
print("Contains __GGUF_PATH__ placeholder:", "__GGUF_PATH__" in modelfile_text)


## Resolve GGUF Host Configuration


In [ ]:
values = {}
if VALUES_PATH.exists():
    values = yaml.safe_load(VALUES_PATH.read_text()) or {}
else:
    print(f"Warning: values file not found: {VALUES_PATH}")

gguf_host_path = values.get("ollamaModel", {}).get("gguf", {}).get("hostPath")
gguf_filename = values.get("ollamaModel", {}).get("gguf", {}).get("filename")
print("GGUF host path:", gguf_host_path)
print("GGUF filename :", gguf_filename)


## Build a Tuned Modelfile (Local Artifact)


In [ ]:
rendered_base = modelfile_text.replace("__GGUF_PATH__", f"/models/gguf/{gguf_filename}")
extra_block = """\nPARAMETER temperature 0.1\nPARAMETER num_ctx 4096\nSYSTEM You are a precise SRE assistant.\n"""
custom_modelfile = rendered_base.strip() + "\n" + extra_block

generated_dir = PROJECT_ROOT / "jupyter-notebooks" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
custom_path = generated_dir / "Modelfile.custom"
custom_path.write_text(custom_modelfile)

print("Wrote:", custom_path)
print(custom_modelfile)

## Optional: Create Tuned Model in Ollama


In [ ]:
RUN_CREATE = False  # Change to True only when you want to create/update the custom model.

if RUN_CREATE:
    payload = {
        "name": CUSTOM_MODEL,
        "modelfile": custom_modelfile,
        "stream": False,
    }
    r = requests.post(f"{OLLAMA_URL}/api/create", json=payload, timeout=600)
    print("status:", r.status_code)
    print(r.text[:2000])
    if r.status_code == 200:
        CUSTOM_MODEL_AVAILABLE = True
else:
    print("Skipped. Set RUN_CREATE=True to call /api/create.")
    print(f"Custom model currently present: {CUSTOM_MODEL_AVAILABLE}")


## Benchmark Base Model (and Tuned Model if Available)


In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

prompts = [
    "Summarize Kubernetes rollout strategy in 4 lines.",
    "Explain a LangSmith trace in simple terms.",
    "Give 3 steps to debug stalled GPU inference.",
]

rows = []
if not OLLAMA_AVAILABLE:
    print("Skipping benchmark: Ollama endpoint is not reachable on localhost:11434")
    print("In another terminal run:")
    print(f"  {OLLAMA_PORT_FORWARD_CMD}")
else:
    available_models = fetch_model_names()
    models = [BASE_MODEL]
    if CUSTOM_MODEL in available_models:
        models.append(CUSTOM_MODEL)
    else:
        print(f"Skipping benchmark for {CUSTOM_MODEL}: model is not present in /api/tags.")
        print("Set RUN_CREATE=True in the previous cell and re-run it if you want a tuned-model comparison.")

    print("Benchmark models:", ", ".join(models))
    for model in models:
        for prompt in prompts:
            body = {
                "model": model,
                "stream": False,
                "messages": [{"role": "user", "content": prompt}],
            }
            started = time.perf_counter()
            try:
                r = requests.post(f"{OLLAMA_URL}/api/chat", json=body, timeout=240)
                latency_ms = (time.perf_counter() - started) * 1000
                rows.append(
                    {
                        "model": model,
                        "prompt": prompt,
                        "status": r.status_code,
                        "latency_ms": round(latency_ms, 2),
                        "ok": r.status_code == 200,
                    }
                )
            except Exception as exc:
                latency_ms = (time.perf_counter() - started) * 1000
                rows.append(
                    {
                        "model": model,
                        "prompt": prompt,
                        "status": -1,
                        "latency_ms": round(latency_ms, 2),
                        "ok": False,
                        "error": str(exc),
                    }
                )

bench_df = pd.DataFrame(rows)
display(bench_df)

if not bench_df.empty:
    ok_df = bench_df[bench_df["ok"]].copy()
    if not ok_df.empty:
        ax = ok_df.groupby("model")["latency_ms"].mean().plot(kind="bar", figsize=(8, 4), title="Average Latency by Model")
        ax.set_ylabel("Latency (ms)")
        plt.xticks(rotation=20)
        plt.show()


## Inspect Cluster Modelfile ConfigMap


In [ ]:
import subprocess

cmd = "kubectl get configmap -n llm-observability ollama-local-modelfile -o yaml"
p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
print(p.stdout if p.stdout else p.stderr)


## Done
You now have a reproducible Modelfile tuning workflow with optional API-based model creation.
